# AutoGen Multi-Agent System with Email Communication via Commune

**Author:** Commune Team  
**Last updated:** 2026-03  
**Estimated time:** 45 minutes  
**Difficulty:** Intermediate to Advanced

---

> **Related notebooks:**
> - `langchain/02-thread-patterns/thread_patterns.ipynb` — deep dive on email threading
> - `notebooks/claude_tool_use_email.ipynb` — Claude tool_use + prompt injection defense
> - `notebooks/structured_extraction.ipynb` — parsing structured data from email replies

## Section 1: Why AutoGen + Commune?

### The Problem: Agents Live in Memory, Customers Don't

AutoGen is excellent at multi-agent coordination. You can have a `ResearchAgent` hand off to a `WriterAgent`, which hands off to a `ReviewerAgent` — all entirely in Python memory, executing in milliseconds.

The problem appears the moment you need an agent to **communicate with someone outside your process**:

- A customer who submitted a support ticket wants a follow-up
- A sales prospect needs a personalized outreach email
- A research agent found relevant contacts and needs to introduce itself
- A human reviewer needs to approve an agent's action before it proceeds

AutoGen has no native mechanism for this. Agents can talk to each other, but the conversation is trapped inside your Python process. The moment you need an email to leave your system — and more importantly, for a reply to **re-enter** your system and trigger the next agent action — you need infrastructure.

That's the gap Commune fills. It gives each agent a real email inbox with:
- A persistent email address (`research-agent@yourdomain.commune.email`)
- Webhook delivery of incoming messages (replies reach your code, not a mailbox)
- Thread continuity (replies link back to the original conversation)
- Structured extraction (parse structured data from reply content without LLM calls)

### Architecture Overview

Here is what you're building:

```
┌─────────────────────────────────────────────────────────────┐
│                    Your Python Process                       │
│                                                              │
│  ┌──────────────────────┐   GroupChat   ┌─────────────────┐ │
│  │    ResearchAgent      │◄────────────►│  OutreachAgent  │ │
│  │  (gather context)     │              │ (write & send)  │ │
│  └──────────┬───────────┘              └────────┬────────┘ │
│             │                                    │          │
│    CommuneEmailTool                    CommuneEmailTool     │
│             │                                    │          │
└─────────────┼────────────────────────────────────┼──────────┘
              │                                    │
              ▼                                    ▼
┌─────────────────────────┐           ┌────────────────────────┐
│  Commune API            │           │  Commune API           │
│  research@yourdomain... │           │  outreach@yourdomain..│
│  (inbox_id: res_xxx)    │           │  (inbox_id: out_yyy)  │
└────────────┬────────────┘           └───────────┬────────────┘
             │                                    │
             │         External Email             │
             │    ┌────────────────────────┐      │
             └───►│   contact@company.com  │◄─────┘
                  │   (real human inbox)   │
                  └──────────┬─────────────┘
                             │  reply via email
                             ▼
              ┌──────────────────────────┐
              │   Commune Webhook        │
              │   POST /webhook/commune  │
              │   (your FastAPI server)  │
              └──────────┬───────────────┘
                         │  triggers next
                         ▼  agent action
              ┌──────────────────────────┐
              │  AutoGen continues       │
              │  GroupChat conversation  │
              └──────────────────────────┘
```

Key design choices visible in this diagram:
1. **Two separate inboxes** — one per agent, not shared
2. **Single webhook endpoint** — distinguishes agents by `inbox_id` in payload
3. **External loop** — agent sends email, waits for webhook, continues AutoGen

### What You'll Build

By the end of this notebook you'll have a working system where:

1. A **ResearchAgent** looks up information about a contact (company, role, recent news)
2. It hands off context to an **OutreachAgent** via AutoGen GroupChat
3. OutreachAgent composes and sends a personalized cold outreach email via Commune
4. When the contact replies, a webhook fires and re-enters the AutoGen conversation
5. OutreachAgent crafts a follow-up and sends it in the **same email thread**

This is a complete human-in-the-loop email workflow driven by multi-agent coordination.

**Prerequisites:**
- A Commune account with API key → [commune.email](https://commune.email)
- A domain configured in Commune (or use the shared `commune.email` testing domain)
- An OpenAI API key (for AutoGen's LLM backbone; swap for any supported provider)

## Section 2: Setup

### Installing Dependencies

We need four packages:
- `commune-mail` — the Commune Python SDK
- `pyautogen` — AutoGen multi-agent framework
- `fastapi` + `uvicorn` — to serve the webhook receiver

If you're running in a virtual environment (recommended), these install cleanly without conflicts.

In [ ]:
%pip install commune-mail pyautogen fastapi uvicorn python-dotenv httpx --quiet

### Environment Variables

This cell configures every credential the system needs. Let's be explicit about what each one does and what breaks if it's missing:

| Variable | Purpose | Where to get it | Missing = |
|----------|---------|----------------|----------|
| `COMMUNE_API_KEY` | Authenticates all Commune API calls | commune.email → Settings → API Keys | `AuthenticationError` on every call |
| `COMMUNE_WEBHOOK_SECRET` | Verifies webhook authenticity (HMAC-SHA256) | Commune inbox settings → Webhooks | Webhook handler can't verify requests |
| `OPENAI_API_KEY` | Powers AutoGen agents' reasoning | platform.openai.com | AutoGen fails to initialize |
| `WEBHOOK_HOST` | Where Commune should POST incoming emails | Your server's public URL | Webhooks never arrive |

> **Production note:** In production, load these from a secrets manager (AWS Secrets Manager, HashiCorp Vault, Railway environment variables). Never hardcode them or commit them to source control. The `.env` approach below is for local development only.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env file if present

# Commune credentials
COMMUNE_API_KEY = os.environ["COMMUNE_API_KEY"]
COMMUNE_WEBHOOK_SECRET = os.environ["COMMUNE_WEBHOOK_SECRET"]

# LLM provider (AutoGen supports many; we use OpenAI here)
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# Where your webhook server is reachable from the internet
# For local dev: use ngrok or cloudflare tunnel
WEBHOOK_HOST = os.environ.get("WEBHOOK_HOST", "https://your-server.example.com")

# Validate everything is present before proceeding
required = [COMMUNE_API_KEY, COMMUNE_WEBHOOK_SECRET, OPENAI_API_KEY]
assert all(required), "One or more required environment variables is missing"

print("Environment configured.")
print(f"Webhook will be registered at: {WEBHOOK_HOST}/webhook/commune")

# Expected output:
# Environment configured.
# Webhook will be registered at: https://your-server.example.com/webhook/commune

## Section 3: Create Agent Inboxes

### One Inbox Per Agent — The Most Important Architectural Decision

Before writing any code, we need to talk about inbox topology. The naive approach is to create one shared inbox and route messages by subject line or To address. **Do not do this.** Here's exactly what breaks:

**The shared inbox failure mode:**

```
Shared inbox: agent@yourdomain.com

Incoming messages:
  1. Re: Research intro (thread_id: th_aaa) → ResearchAgent should handle
  2. Re: Outreach proposal (thread_id: th_bbb) → OutreachAgent should handle
  3. Re: Research intro (thread_id: th_aaa) → ResearchAgent again

Problem: Your webhook receives all three. You now need to:
  - Parse the subject line to guess which agent owns it
  - Hope subjects don't change (they do: "Re: Re: Re: Research intro")
  - Maintain a subject→agent mapping that grows unboundedly
  - Handle collisions when two threads have similar subjects

Result: Routing logic that breaks in production after ~50 threads.
```

**The per-inbox solution:**

```
research-agent@yourdomain.com  → webhook payload always has inbox_id: res_xxx
outreach-agent@yourdomain.com  → webhook payload always has inbox_id: out_yyy

Routing is O(1): switch on inbox_id. Zero ambiguity. Scales to any number of agents.
```

This is the same principle as giving each microservice its own database schema — isolation prevents cross-contamination.

In [ ]:
from commune import CommuneClient

client = CommuneClient(api_key=COMMUNE_API_KEY)

# Create the ResearchAgent inbox
# name is the local part of the email address (before @)
research_inbox = client.inboxes.create(
    name="research-agent",
    display_name="Research Agent",
    # Optional: auto-reply settings, extraction schemas, etc.
)

# Create the OutreachAgent inbox
outreach_inbox = client.inboxes.create(
    name="outreach-agent",
    display_name="Outreach Agent",
)

print(f"Research inbox:  {research_inbox.id}")
print(f"  address: {research_inbox.email_address}")
print()
print(f"Outreach inbox:  {outreach_inbox.id}")
print(f"  address: {outreach_inbox.email_address}")

# Expected output:
# Research inbox:  inb_res1a2b3c4d
#   address: research-agent@yourdomain.commune.email
#
# Outreach inbox:  inb_out5e6f7g8h
#   address: outreach-agent@yourdomain.commune.email

### Configuring Webhooks on Both Inboxes

Webhooks are how Commune tells your code "an email just arrived." Without them, you'd have to poll the API — which adds latency, wastes API quota, and is fragile.

The webhook URL below ends with `/webhook/commune`. We'll implement that endpoint in Section 5. The same URL handles both inboxes — the `inbox_id` in the payload tells us which agent to notify.

> **Production note:** Webhook URLs must be HTTPS. For local development, use [ngrok](https://ngrok.com/) (`ngrok http 8000`) or [Cloudflare Tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/) to expose your local server. The URL must be reachable from Commune's servers, not just your machine.

In [ ]:
WEBHOOK_URL = f"{WEBHOOK_HOST}/webhook/commune"

# Register webhook on research inbox
research_webhook = client.webhooks.create(
    inbox_id=research_inbox.id,
    url=WEBHOOK_URL,
    events=["message.received"],  # fire when an email arrives in this inbox
)

# Register same webhook URL on outreach inbox
# The inbox_id in the payload distinguishes them
outreach_webhook = client.webhooks.create(
    inbox_id=outreach_inbox.id,
    url=WEBHOOK_URL,
    events=["message.received"],
)

print(f"Research webhook: {research_webhook.id} → {WEBHOOK_URL}")
print(f"Outreach webhook: {outreach_webhook.id} → {WEBHOOK_URL}")

# Store inbox IDs for use throughout the notebook
RESEARCH_INBOX_ID = research_inbox.id
OUTREACH_INBOX_ID = outreach_inbox.id

print()
print("Inbox IDs saved to RESEARCH_INBOX_ID and OUTREACH_INBOX_ID")

# Expected output:
# Research webhook: wh_aaa111 → https://your-server.example.com/webhook/commune
# Outreach webhook: wh_bbb222 → https://your-server.example.com/webhook/commune
# Inbox IDs saved to RESEARCH_INBOX_ID and OUTREACH_INBOX_ID

## Section 4: Define AutoGen Agents with Email Capabilities

### The CommuneEmailTool Class

AutoGen agents use tools (functions) to take actions outside their LLM reasoning loop. We'll define a `CommuneEmailTool` class that wraps the three email operations agents need:

1. **`send_email`** — compose and send a new email
2. **`read_latest_email`** — read the most recent message in an inbox
3. **`reply_to_thread`** — reply within an existing email thread

Why a class instead of standalone functions? Because it holds the `CommuneClient` singleton. This is critical — see the Common Mistake cell below.

**Why `thread_id` is optional on `send_email` but mandatory on `reply_to_thread`:**

When you send the *first* email to a contact, there is no thread yet. You omit `thread_id`. Commune creates a new thread and returns `result.thread_id`. From that point forward, every reply to the same contact *must* include that `thread_id`. Without it, each reply becomes a brand-new email — the customer sees 5 separate emails in their inbox instead of one conversation.

In [ ]:
from typing import Optional
from commune import CommuneClient


class CommuneEmailTool:
    """
    Email tool for AutoGen agents. Wraps Commune SDK operations.

    Design note: This class is instantiated ONCE per agent (not per call).
    The client instance is reused across all calls, preserving connection pooling.
    """

    def __init__(self, api_key: str, inbox_id: str):
        # Singleton client — created once, reused for all operations
        self.client = CommuneClient(api_key=api_key)
        self.inbox_id = inbox_id
        self._thread_store: dict[str, str] = {}  # contact_email → thread_id

    def send_email(
        self,
        to: str,
        subject: str,
        body: str,
        thread_id: Optional[str] = None,
    ) -> dict:
        """
        Send an email from this agent's inbox.

        Args:
            to: Recipient email address
            subject: Email subject line
            body: Plain text email body
            thread_id: If continuing an existing conversation, pass the thread_id
                       from the previous message. If None, starts a new thread.

        Returns:
            dict with 'message_id', 'thread_id', and 'status'
        """
        kwargs = {
            "from_inbox_id": self.inbox_id,
            "to": to,
            "subject": subject,
            "text": body,
        }
        if thread_id:
            kwargs["thread_id"] = thread_id

        result = self.client.messages.send(**kwargs)

        # Always store the thread_id for future replies
        self._thread_store[to] = result.thread_id

        return {
            "message_id": result.id,
            "thread_id": result.thread_id,
            "status": "sent",
        }

    def read_latest_email(self) -> Optional[dict]:
        """
        Read the most recent unread message in this agent's inbox.

        Returns:
            dict with 'from', 'subject', 'body', 'thread_id'
            or None if inbox is empty
        """
        messages = self.client.messages.list(
            inbox_id=self.inbox_id,
            limit=1,
        )
        if not messages:
            return None

        msg = messages[0]
        return {
            "from": msg.from_address,
            "subject": msg.subject,
            "body": msg.text or msg.html,
            "thread_id": msg.thread_id,
            "received_at": msg.created_at.isoformat(),
        }

    def reply_to_thread(self, thread_id: str, body: str) -> dict:
        """
        Reply within an existing email thread.

        The thread_id MUST be from a previous message in this conversation.
        This ensures the reply appears in the same thread in the recipient's email client.

        Args:
            thread_id: The thread to reply within (from a previous send result or webhook payload)
            body: Reply body text
        """
        result = self.client.messages.send(
            from_inbox_id=self.inbox_id,
            thread_id=thread_id,
            text=body,
        )
        return {
            "message_id": result.id,
            "thread_id": result.thread_id,
            "status": "replied",
        }

    def get_thread_id_for(self, contact_email: str) -> Optional[str]:
        """Look up the active thread_id for a contact."""
        return self._thread_store.get(contact_email)


print("CommuneEmailTool defined.")

### Common Mistake: Creating a New Client Instance on Every Call

This is the most frequent mistake when wrapping SDK clients in tool functions. It looks harmless in development but causes production failures under load.

> **Production note:** HTTP clients maintain connection pools. Each new client instance creates a new pool. If you create a client on every function call at 100 emails/minute, you'll have 100 open connection pools simultaneously. Most systems have a limit on open file descriptors and socket connections — you'll hit it, and your agent will start throwing `ConnectionError` exceptions.

In [ ]:
# ❌ WRONG — creates a new CommuneClient on every single tool call
def send_email_wrong(to: str, subject: str, body: str) -> dict:
    client = CommuneClient(api_key=COMMUNE_API_KEY)  # NEW CLIENT EVERY CALL
    result = client.messages.send(from_inbox_id="...", to=to, subject=subject, text=body)
    return {"thread_id": result.thread_id}
    # Old client instance is NOT cleaned up — its connection pool stays open
    # At 100 calls/minute: 100 open connection pools after 1 minute
    # At 500 calls/minute: process crashes with "Too many open files"


# ✅ CORRECT — client is created once, reused for all calls (see CommuneEmailTool above)
research_tool = CommuneEmailTool(api_key=COMMUNE_API_KEY, inbox_id=RESEARCH_INBOX_ID)
outreach_tool = CommuneEmailTool(api_key=COMMUNE_API_KEY, inbox_id=OUTREACH_INBOX_ID)

# One singleton per agent. These objects live for the duration of the process.
print("Email tool singletons created for each agent.")
print(f"  research_tool.inbox_id = {research_tool.inbox_id}")
print(f"  outreach_tool.inbox_id = {outreach_tool.inbox_id}")

### Registering Tools with AutoGen Agents

AutoGen's `ConversableAgent` accepts a list of callable functions as tools. The LLM (GPT-4o in this case) decides when to call them based on their docstrings — so docstring quality directly affects tool-use quality.

We define thin wrapper functions that close over the per-agent tool instances. This keeps the tool function signatures clean (AutoGen passes them to the LLM as JSON schema) while routing to the correct inbox.

In [ ]:
import autogen
from autogen import ConversableAgent, GroupChat, GroupChatManager

llm_config = {
    "model": "gpt-4o",
    "api_key": OPENAI_API_KEY,
    "temperature": 0.3,  # Lower temperature = more consistent tool usage decisions
}

# --- Tool wrappers for ResearchAgent ---
# Closures over research_tool (bound to research inbox)

def research_read_email() -> Optional[dict]:
    """Read the latest email received in the research agent's inbox."""
    return research_tool.read_latest_email()

def research_send_email(to: str, subject: str, body: str) -> dict:
    """Send an email from the research agent's inbox to initiate contact."""
    return research_tool.send_email(to=to, subject=subject, body=body)


# --- Tool wrappers for OutreachAgent ---
# Closures over outreach_tool (bound to outreach inbox)

def outreach_send_email(to: str, subject: str, body: str, thread_id: Optional[str] = None) -> dict:
    """
    Send an outreach email from the outreach agent's inbox.
    Pass thread_id to continue an existing conversation thread.
    Omit thread_id when sending the first email to a new contact.
    """
    return outreach_tool.send_email(to=to, subject=subject, body=body, thread_id=thread_id)

def outreach_reply(thread_id: str, body: str) -> dict:
    """Reply to an existing email thread. thread_id is required."""
    return outreach_tool.reply_to_thread(thread_id=thread_id, body=body)


# --- Define the agents ---

research_agent = ConversableAgent(
    name="ResearchAgent",
    system_message=(
        "You are a research agent. Your job is to gather background information about "
        "contacts: their company, role, recent news, and potential pain points. "
        "Summarize your findings concisely and hand off to OutreachAgent. "
        "Use research_read_email to check for any responses that arrived."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

outreach_agent = ConversableAgent(
    name="OutreachAgent",
    system_message=(
        "You are an outreach agent. You receive research context from ResearchAgent and "
        "use it to write personalized, concise outreach emails. "
        "When sending the first email to a contact, omit thread_id. "
        "When replying, always use the thread_id from the original send result. "
        "Never invent a thread_id — only use ones returned by outreach_send_email."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

# Register tools with their respective agents
autogen.register_function(
    research_read_email,
    caller=research_agent,
    executor=research_agent,
    description="Read the latest email in the research inbox",
)
autogen.register_function(
    outreach_send_email,
    caller=outreach_agent,
    executor=outreach_agent,
    description="Send an outreach email. Pass thread_id to continue a thread.",
)
autogen.register_function(
    outreach_reply,
    caller=outreach_agent,
    executor=outreach_agent,
    description="Reply within an existing email thread",
)

print("AutoGen agents initialized with email tools.")

## Section 5: Webhook Handler for Incoming Replies

### How Replies Re-Enter Your AutoGen System

When an external contact replies to an email from your agent, Commune sends an HTTP POST to your webhook URL. This is the bridge between the external email world and your Python process.

There are three rules for webhook handlers that are non-negotiable in production:

1. **Return 200 immediately.** Commune (like all webhook providers) will retry if it doesn't get a 200 within ~30 seconds. If your handler does LLM inference (which can take 5-15 seconds), you'll get duplicate deliveries. The fix: spawn a background task and return 200 instantly.

2. **Verify the HMAC signature.** Without this, any server on the internet can POST fake webhook events to your handler and trigger agent actions. The signature uses the `COMMUNE_WEBHOOK_SECRET` — verify it before parsing the payload.

3. **Implement idempotency.** Webhooks can and will be delivered more than once (network retries, Commune failover). Store `event_id` values you've processed and skip duplicates.

In [ ]:
import asyncio
import hashlib
import hmac
import json
from collections import deque
from fastapi import FastAPI, Request, HTTPException, BackgroundTasks
from fastapi.responses import JSONResponse

app = FastAPI()

# In-memory idempotency store (use Redis in production)
# deque with maxlen prevents unbounded memory growth
processed_event_ids: deque = deque(maxlen=10_000)

# Queue that bridges webhook events to the AutoGen conversation loop
# asyncio.Queue is safe to use across FastAPI background tasks
reply_queue: asyncio.Queue = asyncio.Queue()


def verify_commune_signature(payload: bytes, signature_header: str, secret: str) -> bool:
    """
    Verify the HMAC-SHA256 signature on a Commune webhook request.

    Commune sends: X-Commune-Signature: sha256=<hex_digest>
    We compute our own digest and compare with constant-time comparison
    (hmac.compare_digest) to prevent timing attacks.
    """
    expected_sig = hmac.new(
        secret.encode(),
        payload,
        hashlib.sha256,
    ).hexdigest()

    # Signature header format: "sha256=<hex>"
    provided_sig = signature_header.removeprefix("sha256=")

    # constant-time comparison prevents timing side-channel attacks
    return hmac.compare_digest(expected_sig, provided_sig)


def check_for_prompt_injection(text: str) -> bool:
    """
    Basic prompt injection heuristic check.

    This is a defense-in-depth layer. Commune's API also flags suspected
    injection in message.metadata.prompt_injection_detected.
    See claude_tool_use_email.ipynb for the full security treatment.
    """
    injection_patterns = [
        "ignore previous instructions",
        "ignore all previous",
        "system:",
        "<|im_start|>",
        "forget your instructions",
        "new instructions:",
    ]
    lowered = text.lower()
    return any(pattern in lowered for pattern in injection_patterns)


async def process_reply_in_background(event: dict) -> None:
    """
    Background task: process an incoming email reply and push it to the reply queue.
    The AutoGen conversation loop reads from this queue to continue the conversation.
    """
    payload = event["payload"]
    message = payload.get("message", {})
    body_text = message.get("text", "")

    # Security: reject suspected injection attempts before any LLM processing
    if check_for_prompt_injection(body_text):
        print(f"[SECURITY] Potential prompt injection in message {message.get('id')} — discarded")
        return

    # Also check Commune's own injection detection metadata
    metadata = message.get("metadata", {})
    if metadata.get("prompt_injection_detected"):
        print(f"[SECURITY] Commune flagged message {message.get('id')} — discarded")
        return

    await reply_queue.put({
        "inbox_id": payload.get("inbox_id"),
        "from": message.get("from"),
        "thread_id": message.get("thread_id"),
        "body": body_text,
        "subject": message.get("subject", ""),
    })


@app.post("/webhook/commune")
async def commune_webhook(
    request: Request,
    background_tasks: BackgroundTasks,
) -> JSONResponse:
    """
    Commune webhook receiver.

    CRITICAL: This handler returns 200 IMMEDIATELY after:
      1. Signature verification (fast — pure crypto)
      2. Idempotency check (fast — in-memory lookup)

    All actual processing (LLM calls, database writes, AutoGen continuation)
    happens in a background task AFTER the 200 is returned.
    """
    raw_body = await request.body()

    # 1. Verify signature — reject unauthenticated requests immediately
    signature = request.headers.get("X-Commune-Signature", "")
    if not verify_commune_signature(raw_body, signature, COMMUNE_WEBHOOK_SECRET):
        raise HTTPException(status_code=401, detail="Invalid webhook signature")

    event = json.loads(raw_body)

    # 2. Idempotency check — skip events we've already processed
    event_id = event.get("id")
    if event_id in processed_event_ids:
        # Return 200 so Commune doesn't retry, but do no processing
        return JSONResponse({"status": "duplicate", "event_id": event_id})
    processed_event_ids.append(event_id)

    # 3. Return 200 NOW — before any slow processing
    # Background task runs AFTER response is sent
    background_tasks.add_task(process_reply_in_background, event)

    return JSONResponse({"status": "received", "event_id": event_id})


print("FastAPI webhook handler defined.")
print("Start the server with: uvicorn <module>:app --host 0.0.0.0 --port 8000")

### What Happens If You Don't Return 200 Immediately?

This is worth dwelling on because the failure mode is subtle and hard to debug in production:

```
Scenario: Your webhook handler does LLM inference synchronously.

Timeline:
  T+0s:  Contact replies to outreach email
  T+0s:  Commune sends POST to your webhook
  T+8s:  Your handler: GPT-4o generates AutoGen response
  T+10s: Your handler: AutoGen sends follow-up email
  T+12s: Your handler returns 200

Commune's perspective:
  T+0s:  POST sent
  T+30s: Timeout! No 200 received within 30 seconds.
         [Note: Your 200 arrived, but Commune already gave up and is retrying]
  T+30s: Retry #1 POST sent
  T+30s: Your handler processes the same event AGAIN
  T+38s: Another AutoGen response generated
  T+40s: Another follow-up email sent to the contact

Result: Contact receives the same follow-up email twice.
        Your agent looks broken (and unprofessional).
```

The idempotency check protects against this, but the best defense is returning 200 fast in the first place.

## Section 6: Run the Multi-Agent Email Workflow

### Initializing the GroupChat

AutoGen's GroupChat coordinates which agent speaks next. We use `speaker_selection_method="auto"` so GPT-4 decides the ordering based on conversation context — ResearchAgent goes first (it has the contact info), then OutreachAgent takes over once research is complete.

In [ ]:
# Contact we're reaching out to (in a real system, this comes from a CRM or leads list)
CONTACT_EMAIL = "alex.johnson@acmecorp.example.com"
CONTACT_NAME = "Alex Johnson"
CONTACT_COMPANY = "Acme Corp"
CONTACT_ROLE = "VP of Engineering"

group_chat = GroupChat(
    agents=[research_agent, outreach_agent],
    messages=[],
    max_round=10,
    speaker_selection_method="auto",
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

print("GroupChat initialized with ResearchAgent and OutreachAgent.")

### Running the Outreach Workflow

We initiate the conversation with a task description. ResearchAgent will gather context (in a real system it would call a web search or CRM API — here we give it inline context for clarity), then hand off to OutreachAgent which will send the actual email via Commune.

Watch the agent messages below — the AutoGen conversation is happening in memory between the agents, but the actual email goes out through Commune's infrastructure and arrives in the contact's real inbox.

In [ ]:
initial_task = f"""
We need to reach out to {CONTACT_NAME}, {CONTACT_ROLE} at {CONTACT_COMPANY}.

Research context (already gathered):
- Acme Corp recently announced a 3x increase in their support ticket volume
- They are migrating to a microservices architecture (per their engineering blog)
- Alex specifically wrote about the pain of agent-to-human escalation in their system
- They are likely evaluating tools to help their AI agents communicate via email

ResearchAgent: Summarize the key pain points and what angle our outreach should take.
OutreachAgent: Write and send a personalized email to {CONTACT_EMAIL} using the outreach_send_email tool.
The email should be concise (under 150 words), reference a specific pain point, and ask for a 20-minute call.
"""

# This initiates the multi-agent conversation
# The conversation will:
#   1. ResearchAgent summarizes the research
#   2. OutreachAgent composes and calls outreach_send_email tool
#   3. Commune API sends the email
#   4. GroupChat concludes

result = research_agent.initiate_chat(
    manager,
    message=initial_task,
)

# The thread_id from the sent email is stored in outreach_tool._thread_store
active_thread_id = outreach_tool.get_thread_id_for(CONTACT_EMAIL)
print(f"\nEmail sent. Active thread_id: {active_thread_id}")
print("Waiting for reply via webhook...")

# Expected output (abbreviated):
# ResearchAgent (to chat_manager):
#   Key pain points for Alex Johnson at Acme Corp:
#   1. 3x support volume increase = agent systems under strain
#   2. Microservices migration = agent-to-agent communication complexity
#   3. Alex personally cares about agent-to-human escalation
#   Recommended angle: position Commune as the missing link...
#
# OutreachAgent (to chat_manager):
#   I'll send a personalized email now.
#   [Calls outreach_send_email tool]
#
# Email sent. Active thread_id: th_abc123def456

### The Email as Received by the Contact

After the OutreachAgent calls `outreach_send_email`, the contact receives something like this in their inbox:

```
From:    Outreach Agent <outreach-agent@yourdomain.commune.email>
To:      alex.johnson@acmecorp.example.com
Subject: Agent-to-human escalation at Acme Corp

Hi Alex,

I read your engineering blog post about the friction in your agent-to-human
escalation flow — specifically the part about agents needing to "reach outside
their memory" to notify humans.

We built Commune to solve exactly that: real email inboxes for AI agents, with
thread continuity so replies land back in the right agent's context.

Given your 3x ticket volume increase, worth a 20-minute call to see if it fits?

Best,
Outreach Agent (on behalf of the team)
```

Notice:
- The From address is the Commune inbox address — replies go back to Commune, not a human mailbox
- The subject is specific to Alex, not a generic template
- Under 150 words as specified in the task

### Handling the Reply: Webhook → AutoGen Continuation

When Alex replies, the webhook fires. We read from the `reply_queue` (populated by the webhook handler) and use the `thread_id` to continue the conversation in the same email thread.

This is the crucial reconnection moment: an external email re-enters your AutoGen process and drives the next agent action.

In [ ]:
import asyncio

async def handle_incoming_reply():
    """
    Wait for an email reply to arrive via webhook, then continue the AutoGen conversation.

    In production, this would be a long-running async loop, not a one-shot await.
    See the 'Production Considerations' section for the event-driven architecture pattern.
    """
    print("Waiting for reply (webhook will populate reply_queue)...")

    # In a real server, this await suspends the coroutine until a reply arrives
    # asyncio.wait_for adds a timeout so we don't wait forever in a notebook
    try:
        reply = await asyncio.wait_for(reply_queue.get(), timeout=300.0)  # 5 min timeout
    except asyncio.TimeoutError:
        print("No reply received within 5 minutes.")
        return

    print(f"Reply received from: {reply['from']}")
    print(f"Thread ID: {reply['thread_id']}")
    print(f"Content: {reply['body'][:200]}")

    # CRITICAL: use the thread_id from the reply, not from our send result
    # They should match — but always trust the incoming payload, not your local state
    incoming_thread_id = reply["thread_id"]

    # Continue the AutoGen conversation with the reply content
    followup_task = f"""
    We received a reply from {CONTACT_EMAIL}:

    \"{reply['body']}\"

    The reply is in thread_id: {incoming_thread_id}

    OutreachAgent: Write and send a follow-up reply in the SAME thread.
    Use the outreach_reply tool with thread_id={incoming_thread_id}.
    Keep it under 100 words. Move toward scheduling a call.
    """

    result = outreach_agent.initiate_chat(
        manager,
        message=followup_task,
        clear_history=False,  # preserve conversation context
    )

    print("Follow-up sent in the same thread.")
    print(f"Alex sees one email conversation, not two separate emails.")


# In a Jupyter notebook, run the async function with:
# asyncio.run(handle_incoming_reply())
# Or in an async context:
# await handle_incoming_reply()

print("Reply handler defined. Call handle_incoming_reply() when a reply is expected.")
print("The webhook handler populates reply_queue automatically.")

## Section 7: Production Considerations

### Idempotency: What Happens When Commune Retries a Webhook?

Commune will retry webhook delivery if your server is temporarily unavailable (deploys, restarts, brief outages). Every retry delivers the same event — same `event_id`, same payload.

**The failure mode without idempotency:** Your agent sends the same reply email twice (or more). The contact receives duplicates. If the action involves sending money, creating records, or external API calls, duplicates can be costly.

**The fix:** Store every processed `event_id`. Before processing, check if you've seen this ID. If yes, return 200 and do nothing.

The in-memory `deque(maxlen=10_000)` in the webhook handler above works for single-process deployments. For multi-process deployments (multiple server instances), use Redis:

```python
# Redis-backed idempotency (production pattern)
import redis
r = redis.Redis(host='localhost', port=6379)

def is_duplicate(event_id: str) -> bool:
    # SET if Not eXists, with 24-hour TTL
    # Returns True if key was just set (new), False if it already existed (duplicate)
    was_new = r.set(f"webhook:{event_id}", "1", nx=True, ex=86400)
    return not was_new
```

### Scaling: What Breaks at 1,000 Emails/Day?

At 1,000 emails/day (~1 per minute), the current architecture is fine. At 10,000/day, watch for:

1. **AutoGen GroupChat bottleneck** — each email reply re-initiates a GroupChat round. At high volume, these queue up. Solution: use a job queue (Celery, RQ, or ARQ for async) instead of processing webhook replies synchronously.

2. **LLM API rate limits** — OpenAI limits tokens per minute. At high volume, AutoGen's multi-agent conversation will hit rate limits. Solution: implement exponential backoff, or use a local model (Ollama + Mistral) for low-stakes agents.

3. **`reply_queue` memory growth** — the in-memory asyncio queue will grow if replies arrive faster than the agent processes them. Solution: switch to Redis Streams or a managed queue service.

### Monitoring: How Do You Know If Agent Emails Are Being Delivered?

Email deliverability is a silent failure: your API call succeeds, but the email lands in the recipient's spam folder or bounces. Monitor for:

- **Bounce events** — configure Commune to send `message.bounced` webhook events. Log these and remove bounced addresses from your contact list.
- **Open rates** — Commune can report when emails are opened. A 0% open rate after 100 sends is a signal of deliverability issues.
- **Reply rate** — if you're sending outreach and getting 0 replies, check spam placement before blaming the copy.
- **Commune credits** — monitor your API credit usage. A sudden spike means unexpected email volume (possibly a bug looping sends).

## Section 8: What We Built

This notebook built a complete human-in-the-loop multi-agent email system:

1. **Two isolated agent inboxes** — one per agent, routing by `inbox_id` instead of fragile subject-line matching
2. **CommuneEmailTool** — a singleton-client wrapper that AutoGen agents can call as a tool, with connection pooling handled correctly
3. **A FastAPI webhook handler** — with HMAC verification, immediate 200 response, background processing, idempotency, and prompt injection checks
4. **AutoGen GroupChat** — ResearchAgent → OutreachAgent handoff, initiated by a task description
5. **Reply loop** — webhook fires when contact replies, thread_id ensures the reply lands in the same conversation

**Next steps:**
- Add structured extraction to parse meeting preferences from replies → `notebooks/structured_extraction.ipynb`
- Deepen the security treatment (prompt injection defense) → `notebooks/claude_tool_use_email.ipynb`
- Understand email thread continuity deeply → `langchain/02-thread-patterns/thread_patterns.ipynb`
- Deploy the webhook handler to production → see `AGENTS.md` for Railway deployment commands

**Key decisions made in this notebook and why they matter:**

| Decision | Why |
|----------|-----|
| One inbox per agent | Routing by `inbox_id` is O(1) and unambiguous; subject-line routing fails after ~50 threads |
| Singleton CommuneClient | Prevents connection pool exhaustion under load |
| Return 200 before processing | Prevents duplicate webhook delivery from Commune retries |
| HMAC verification | Prevents unauthorized webhook injection attacks |
| thread_id on all replies | Ensures replies appear in the same thread in recipient's email client |
| asyncio.Queue for reply bridging | Decouples webhook ingestion from AutoGen processing; allows backpressure |